The Chinook record store has just signed a deal with a new record label, and you've been tasked with selecting the first three albums that will be added to the store, from a list of four. All four albums are by artists that don't have any tracks in the store right now - we have the artist names, and the genre of music they produce:

Artist Name	Genre
Regal	Hip-Hop
Red Tone	Punk
Meteor and the Girls	Pop
Slim Jim Bites	Blues

The record label specializes in artists from the USA, and they have given Chinook some money to advertise the new albums in the USA, so we're interested in finding out which genres sell the best in the USA.

In [1]:
%%capture
%load_ext sql
%sql sqlite:///chinook.db

In [2]:
%%sql 
SELECT name,type FROM sqlite_master WHERE type IN ('table', 'view');

 * sqlite:///chinook.db
Done.


name,type
album,table
artist,table
customer,table
employee,table
genre,table
invoice,table
invoice_line,table
media_type,table
playlist,table
playlist_track,table


In [4]:
%%sql
WITH qty_per_genre AS 
(
    SELECT g.name genre_name, il.quantity
        FROM genre g
        INNER JOIN track t ON t.genre_id=g.genre_id
        INNER JOIN invoice_line il ON il.track_id=t.track_id
        INNER JOIN invoice i ON i.invoice_id=il.invoice_id
        WHERE i.billing_country='USA'
)

SELECT genre_name, SUM(quantity) num_tracks_sold,
        CAST(SUM(quantity) AS Float)/(SELECT COUNT(*) FROM qty_per_genre)
        AS percentage_market_share
    FROM qty_per_genre
    GROUP BY genre_name
    ORDER BY 2 DESC

 * sqlite:///chinook.db
Done.


genre_name,num_tracks_sold,percentage_market_share
Rock,561,0.5337773549000951
Alternative & Punk,130,0.12369172216936251
Metal,124,0.11798287345385347
R&B/Soul,53,0.05042816365366318
Blues,36,0.03425309229305423
Alternative,35,0.03330161750713606
Latin,22,0.02093244529019981
Pop,22,0.02093244529019981
Hip Hop/Rap,20,0.019029495718363463
Jazz,14,0.013320647002854425


Each customer for the Chinook store gets assigned to a sales support agent within the company when they first make a purchase. You have been asked to analyze the purchases of customers belonging to each employee to see if any sales support agent is performing either better or worse than the others.

You might like to consider whether any extra columns from the employee table explain any variance you see, or whether the variance might instead be indicative of employee performance.

In [6]:
%%sql
SELECT 
    e.employee_id, e.title, e.country,
    SUM(i.total) expenses_by_customers
    
    FROM employee e
    LEFT JOIN customer c ON e.employee_id=c.support_rep_id
    LEFT JOIN invoice i ON i.customer_id=c.customer_id
    
    GROUP BY 1
    

 * sqlite:///chinook.db
Done.


employee_id,title,country,expenses_by_customers
1,General Manager,Canada,None
2,Sales Manager,Canada,None
3,Sales Support Agent,Canada,1731.510000000004
4,Sales Support Agent,Canada,1584.0000000000032
5,Sales Support Agent,Canada,1393.9200000000028
6,IT Manager,Canada,None
7,IT Staff,Canada,None
8,IT Staff,Canada,None
